In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from pymongo import MongoClient

NOMBRE_GRUPO = "G9-AgroTech"

# --- PASO 1: CONFIGURACIÓN ---
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# --- PASO 2: SCRAPING ---
datos_finales = []
limite_paginas = 3

try:
    driver.get("http://books.toscrape.com/")
    
    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Nivel de Profundidad {nivel_pagina + 1} ---")
        
        # 1. Espera al Contenedor Raíz
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "product_pod"))
        )

        # 2. Captura de Bloques
        bloques_primer_nivel = driver.find_elements(By.CLASS_NAME, "product_pod")
        
        for bloque in bloques_primer_nivel:
            # 3. Búsqueda Detallada
            dato_identificador = bloque.find_element(By.TAG_NAME, "h3").find_element(By.TAG_NAME, "a").get_attribute("title")
            dato_valor = bloque.find_element(By.CLASS_NAME, "price_color").text
            
            item_extraido = {
                "identificador": dato_identificador,
                "valor": dato_valor,
                "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                "grupo": NOMBRE_GRUPO
            }
            datos_finales.append(item_extraido)

        # --- NAVEGACIÓN (Paginación) ---
        try:
            disparador_siguiente = driver.find_element(By.CSS_SELECTOR, "li.next a")
            disparador_siguiente.click()
        except Exception:
            print("Fin del árbol de navegación o nodo no encontrado.")
            break 

    print(f"\nExtracción finalizada. Registros en memoria: {len(datos_finales)}")

except Exception as e:
    print(f"Fallo en la arquitectura de scraping: {e}")

finally:
    driver.quit()


# --- PASO 3: BASE DE DATOS ---
try:
    # 'database' debe coincidir con el nombre del servicio en tu docker-compose
    client = MongoClient('database', 27017)
    db = client['proyecto_bigdata']
    coleccion = db['datos_scraping2']
    print("\nCONEXION ESTABLECIDA: Conectado a MongoDB exitosamente.")

    registros_exitosos = 0
    registros_fallidos = 0

    for dato in datos_finales:
        try:
            # Limpieza y conversión
            valor_sucio = str(dato.get('valor', '0'))
            valor_limpio = valor_sucio.replace('£', '').replace(',', '').strip()
            
            dato['valor'] = float(valor_limpio)
            
            # Inserción
            coleccion.insert_one(dato)
            registros_exitosos += 1
            
        except ValueError:
            print("ADVERTENCIA: No se pudo procesar el valor:", valor_sucio)
            registros_fallidos += 1
        except Exception as e:
            print("ERROR EN INSERCION:", e)
            registros_fallidos += 1

    print("\nRESUMEN DE CARGA:")
    print(f"Registros exitosos: {registros_exitosos}")
    print(f"Registros fallidos: {registros_fallidos}")

except Exception as e:
    print("ERROR DE CONEXION A MONGO:", e)

--- Procesando Nivel de Profundidad 1 ---
--- Procesando Nivel de Profundidad 2 ---
--- Procesando Nivel de Profundidad 3 ---

Extracción finalizada. Registros en memoria: 60

CONEXION ESTABLECIDA: Conectado a MongoDB exitosamente.

RESUMEN DE CARGA:
Registros exitosos: 60
Registros fallidos: 0


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. Iniciar Sesión de Spark (Asegúrate de que el conector sea la versión correcta)
spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping2") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()

# 2. Cargar los datos
try:
    df = spark.read.format("mongodb").load()
    
    # 3. Limpieza corregida:
    # Cambiamos "item" por "identificador", que es como lo guardaste en Selenium
    df_filtrado = df.filter(
        (col("identificador").isNotNull()) & 
        (col("valor") > 0)
    )

    print("PASO 3: Limpieza completada.")
    print(f"Registros originales: {df.count()}")
    print(f"Registros válidos: {df_filtrado.count()}")
    
    # Mostrar una muestra para confirmar
    df_filtrado.show(5)

except Exception as e:
    print(f"Error al procesar datos con Spark: {e}")

PASO 3: Limpieza completada.
Registros originales: 60
Registros válidos: 60
+--------------------+-------------------+-----------+--------------------+-----+
|                 _id|      fecha_captura|      grupo|       identificador|valor|
+--------------------+-------------------+-----------+--------------------+-----+
|69d65479c216617a6...|2026-04-08 13:13:25|G9-AgroTech|A Light in the Attic|51.77|
|69d65479c216617a6...|2026-04-08 13:13:25|G9-AgroTech|  Tipping the Velvet|53.74|
|69d65479c216617a6...|2026-04-08 13:13:25|G9-AgroTech|          Soumission| 50.1|
|69d65479c216617a6...|2026-04-08 13:13:25|G9-AgroTech|       Sharp Objects|47.82|
|69d65479c216617a6...|2026-04-08 13:13:25|G9-AgroTech|Sapiens: A Brief ...|54.23|
+--------------------+-------------------+-----------+--------------------+-----+
only showing top 5 rows



In [5]:
df.select("identificador", "valor").show()

+--------------------+-----+
|       identificador|valor|
+--------------------+-----+
|A Light in the Attic|51.77|
|  Tipping the Velvet|53.74|
|          Soumission| 50.1|
|       Sharp Objects|47.82|
|Sapiens: A Brief ...|54.23|
|     The Requiem Red|22.65|
|The Dirty Little ...|33.34|
|The Coming Woman:...|17.93|
|The Boys in the B...| 22.6|
|     The Black Maria|52.15|
|Starving Hearts (...|13.99|
|Shakespeare's Son...|20.66|
|         Set Me Free|17.46|
|Scott Pilgrim's P...|52.29|
|Rip it Up and Sta...|35.02|
|Our Band Could Be...|57.25|
|                Olio|23.88|
|Mesaerion: The Be...|37.59|
|Libertarianism fo...|51.33|
|It's Only the Him...|45.17|
+--------------------+-----+
only showing top 20 rows



In [6]:
df.filter(df["valor"] > 47).show()

+--------------------+-------------------+-----------+--------------------+-----+
|                 _id|      fecha_captura|      grupo|       identificador|valor|
+--------------------+-------------------+-----------+--------------------+-----+
|69d65479c216617a6...|2026-04-08 13:13:25|G9-AgroTech|A Light in the Attic|51.77|
|69d65479c216617a6...|2026-04-08 13:13:25|G9-AgroTech|  Tipping the Velvet|53.74|
|69d65479c216617a6...|2026-04-08 13:13:25|G9-AgroTech|          Soumission| 50.1|
|69d65479c216617a6...|2026-04-08 13:13:25|G9-AgroTech|       Sharp Objects|47.82|
|69d65479c216617a6...|2026-04-08 13:13:25|G9-AgroTech|Sapiens: A Brief ...|54.23|
|69d65479c216617a6...|2026-04-08 13:13:26|G9-AgroTech|     The Black Maria|52.15|
|69d65479c216617a6...|2026-04-08 13:13:26|G9-AgroTech|Scott Pilgrim's P...|52.29|
|69d65479c216617a6...|2026-04-08 13:13:26|G9-AgroTech|Our Band Could Be...|57.25|
|69d65479c216617a6...|2026-04-08 13:13:26|G9-AgroTech|Libertarianism fo...|51.33|
|69d65479c216617

In [7]:
df.groupBy("grupo").count().show()

+-----------+-----+
|      grupo|count|
+-----------+-----+
|G9-AgroTech|   60|
+-----------+-----+



In [9]:
from pyspark.sql import functions as F

# 1. Usamos 'identificador' (que es el nombre real en tu MongoDB)
# Agrupamos por 'grupo' ya que 'categoria' no existe en tu extracción actual
reporte_grupos = df.groupBy("grupo").agg(
    F.count("identificador").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR GRUPO (Basado en datos disponibles):")
reporte_grupos.show()

ANALISIS DE MERCADO POR GRUPO (Basado en datos disponibles):
+-----------+---------------+-----------------+-------------+-------------+
|      grupo|cantidad_libros|  precio_promedio|precio_minimo|precio_maximo|
+-----------+---------------+-----------------+-------------+-------------+
|G9-AgroTech|             60|35.00266666666666|        12.84|        57.31|
+-----------+---------------+-----------------+-------------+-------------+



In [12]:
from pyspark.sql import functions as F

# 1. Ordenar por valor de forma descendente
# 2. Tomar el primer registro
ganador = df.orderBy(F.desc("valor")).limit(1)

ganador.select("identificador", "valor", "grupo", "fecha_captura").show()

+--------------------+-----+-----------+-------------------+
|       identificador|valor|      grupo|      fecha_captura|
+--------------------+-----+-----------+-------------------+
|Slow States of Co...|57.31|G9-AgroTech|2026-04-08 13:13:28|
+--------------------+-----+-----------+-------------------+



In [16]:
from pyspark.sql import functions as F

# 1. DEFINIR LA VARIABLE (Esto es lo que falta)
NOMBRE_GRUPO = "G9-AgroTech" 

# 2. CONSULTA DE SALIDA
ticket_salida = df.filter(F.col("grupo") == NOMBRE_GRUPO) \
    .groupBy("grupo") \
    .agg(
        F.count("identificador").alias("total_libros"),
        F.round(F.avg("valor"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print(f"--- TICKET DE SALIDA: {NOMBRE_GRUPO} ---")
ticket_salida.show()



--- TICKET DE SALIDA: G9-AgroTech ---
+-----------+------------+------------+---------------------+
|      grupo|total_libros|precio_medio|ultima_sincronizacion|
+-----------+------------+------------+---------------------+
|G9-AgroTech|          60|        35.0|  2026-04-08 13:13:28|
+-----------+------------+------------+---------------------+

